# 2. Sygnały i falki jako wektory

**Arkadiusz Liber** · *Falki w praktyce. Od wzoru do kodu*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/writingin/falki-w-praktyce-od-wzoru-do-kodu-notatniki/blob/main/rozdzial-02.ipynb)

Notatnik do książki. Zawiera wszystkie programy z tekstu rozdziału oraz ćwiczenia z rozwiązaniami.

Notatnik działa lokalnie (Jupyter) i w Google Colab. Komórki można uruchamiać po kolei poleceniem *Uruchom wszystko*.

In [1]:
# Uruchom tę komórkę jako pierwszą (lokalnie albo w Google Colab).
import sys, subprocess, importlib.util, os
if importlib.util.find_spec("pywt") is None:      # PyWavelets
    subprocess.check_call([sys.executable, "-m", "pip",
                           "install", "-q", "PyWavelets"])
import numpy as np
import matplotlib.pyplot as plt
import pywt

SZARY, NIEBIESKI, RDZAWY, ZIELONY = "#757575", "#1273E2", "#D93B0B", "#078832"
FIOLET, BURSZTYN = "#9258D5", "#B36000"
plt.rcParams.update({
    "text.color": SZARY, "axes.labelcolor": SZARY, "axes.edgecolor": SZARY,
    "xtick.color": SZARY, "ytick.color": SZARY, "axes.titlecolor": SZARY,
    "font.size": 10, "axes.titlesize": 10,
    "axes.spines.top": False, "axes.spines.right": False})


## Programy z tekstu rozdziału

### Kod 2.1. Sprawdzenie aksjomatów grupy dla dodawania modulo 5

In [2]:
import itertools

n = 5
A = range(n)
dodaj = lambda a, b: (a + b) % n

laczne = all(
    dodaj(dodaj(a, b), c) == dodaj(a, dodaj(b, c))
    for a, b, c in itertools.product(A, repeat=3))
neutralny = [e for e in A
             if all(dodaj(a, e) == a for a in A)]
odwrotne = {a: next(b for b in A if dodaj(a, b) == 0)
            for a in A}

print("łączność:", laczne)
print("element neutralny:", neutralny)
print("elementy odwrotne:", odwrotne)

łączność: True
element neutralny: [0]
elementy odwrotne: {0: 0, 1: 4, 2: 3, 3: 2, 4: 1}


### Kod 2.2. Sygnały jako wektory w NumPy

In [3]:
import numpy as np

t = np.arange(8) / 8
v1 = np.sin(2*np.pi*t)        # sygnał 1
v2 = np.cos(2*np.pi*t)        # sygnał 2

suma = v1 + v2                # dodawanie wektorów
wzmocniony = 2.5 * v1         # mnożenie przez skalar

# rozdzielność mnożenia względem dodawania
print(np.allclose(2.5*(v1 + v2), 2.5*v1 + 2.5*v2))

True


### Kod 2.3. Grupa przesunięć cyklicznych: zespolone sinusoidy są wektorami własnymi każdego filtru okresowego

In [4]:
import numpy as np

N = 8
S = np.roll(np.eye(N), 1, axis=0)          # przesunięcie o 1 próbkę
print(np.allclose(np.linalg.matrix_power(S, N), np.eye(N)))

# filtr okresowy: kombinacja liniowa przesunięć
h = np.array([0.5, 0.3, 0.2])
C = sum(h[m]*np.linalg.matrix_power(S, m) for m in range(3))

n = np.arange(N)
for k in (0, 1, 3):
    chi = np.exp(2j*np.pi*k*n/N)           # homomorfizm (2.5)
    wartosc = (C @ chi)[0]/chi[0]          # wartość własna
    print(k, np.allclose(C @ chi, wartosc*chi), np.round(wartosc, 3))

# wartości własne filtru: transformata Fouriera jego współczynników
print(np.round(np.fft.fft(h, N)[[0, 1, 3]], 3))

True
0 True (1+0j)
1 True (0.712-0.412j)
3 True (0.288-0.012j)
[1.   +0.j    0.712-0.412j 0.288-0.012j]


### Kod 2.4. Rozkład na część parzystą i nieparzystą oraz dopasowanie korzystające z symetrii

In [5]:
import numpy as np
import sympy as sp

x = sp.symbols("x")
f = sp.exp(x)
f_p = ((f + f.subs(x, -x))/2).rewrite(sp.cosh).simplify()
f_n = ((f - f.subs(x, -x))/2).rewrite(sp.sinh).simplify()
print(f_p, "|", f_n)
print(sp.integrate(f_p*f_n, (x, -1, 1)))     # ortogonalność

# dopasowanie funkcji parzystej wielomianem stopnia 6
t = np.linspace(-1, 1, 201)
y = np.cosh(2*t)
pelne = np.polynomial.polynomial.polyfit(t, y, 6)
V = np.stack([t**0, t**2, t**4, t**6], axis=1)   # potęgi parzyste
parzyste = np.linalg.lstsq(V, y, rcond=None)[0]
print(np.round(pelne, 4))
print(np.round(parzyste, 4))

cosh(x) | sinh(x)
0
[ 1.      0.      2.0014  0.      0.6591 -0.      0.1016]
[1.     2.0014 0.6591 0.1016]


### Kod 2.5. Grupa afiniczna: składanie skalowania z przesunięciem

In [6]:
from fractions import Fraction as F

def zloz(g, h):                       # wzór (2.7)
    return (g[0]*h[0], g[0]*h[1] + g[1])

def odwrotny(g):
    return (1/g[0], -g[1]/g[0])

g, h, k = (F(2), F(3)), (F(1, 2), F(-1)), (F(4), F(5))
print(zloz(zloz(g, h), k) == zloz(g, zloz(h, k)))      # łączność
print(zloz(g, odwrotny(g)) == (1, 0))                  # odwrotność
print(zloz(g, h) == zloz(h, g))                        # przemienność?

# czy zbiór par (2^j, k 2^j) jest zamknięty na składanie?
def diadyczny(g):
    a, b = g
    potega = a.numerator & (a.numerator - 1) == 0 and \
             a.denominator & (a.denominator - 1) == 0
    return potega and (b/a).denominator == 1

u, v = (F(1), F(1)), (F(2), F(2))
w = zloz(u, v)
print(diadyczny(u), diadyczny(v), w, diadyczny(w))

True
True
False
True True (Fraction(2, 1), Fraction(3, 1)) False


### Kod 2.6. Interpolacja Lagrange'a w ciele skończonym: odtwarzanie utraconych danych

In [7]:
p = 257                                # liczba pierwsza: ciało Z_257

def lagrange(punkty, x0):
    """Wartość w x0 wielomianu przechodzącego przez punkty (x, y)."""
    wynik = 0
    for i, (xi, yi) in enumerate(punkty):
        licznik = mianownik = 1
        for j, (xj, _) in enumerate(punkty):
            if i != j:
                licznik = licznik*(x0 - xj) % p
                mianownik = mianownik*(xi - xj) % p
        wynik += yi*licznik*pow(mianownik, -1, p)   # dzielenie w ciele
    return wynik % p

dane = [ord(z) for z in "FALKI"]       # 5 porcji danych
punkty = list(enumerate(dane, start=1))
wyslane = punkty + [(x, lagrange(punkty, x)) for x in range(6, 10)]
print(wyslane)

odebrane = [wyslane[i] for i in (1, 3, 5, 6, 8)]    # cztery zaginęły
odtworzone = [lagrange(odebrane, x) for x in range(1, 6)]
print("".join(chr(v) for v in odtworzone))

[(1, 70), (2, 65), (3, 76), (4, 75), (5, 73), (6, 120), (7, 48), (8, 242), (9, 98)]
FALKI


### Kod 2.7. Liczby zmiennoprzecinkowe nie są ciałem, a liczby wymierne w SymPy są

In [8]:
import sympy as sp

print((0.1 + 0.2) + 0.3 == 0.1 + (0.2 + 0.3))          # łączność?
a, b, c = sp.Rational(1, 10), sp.Rational(2, 10), sp.Rational(3, 10)
print((a + b) + c == a + (b + c), (a + b) + c)

# macierz odwrotna obliczona dokładnie, w liczbach wymiernych
H = sp.Matrix(3, 3, lambda i, j: sp.Rational(1, i + j + 1))
print(H.inv())

False
True 3/5
Matrix([[9, -36, 30], [-36, 192, -180], [30, -180, 180]])


### Kod 2.8. Wielomiany jako wektory współczynników

In [9]:
import numpy as np
from numpy.polynomial import Polynomial

p = Polynomial([1, 0, -2])           # 1 - 2x^2
q = Polynomial([0, 3, 0, 1])         # 3x + x^3

s = 2*p + q                          # kombinacja liniowa wektorów
print(s.coef)                        # współrzędne wyniku
print(np.isclose(s(0.7), 2*p(0.7) + q(0.7)))

# te same działania na samych wektorach współczynników
vp, vq = np.array([1, 0, -2, 0.]), np.array([0, 3, 0, 1.])
print(2*vp + vq)

[ 2.  3. -4.  1.]
True
[ 2.  3. -4.  1.]


### Kod 2.9. Norma, odległość, iloczyn skalarny i kąt między sygnałami

In [10]:
import numpy as np

t = np.arange(64) / 64
v1 = np.sin(2*np.pi*3*t)
v2 = np.sin(2*np.pi*3*t + np.pi/3)   # przesunięty w fazie
v3 = np.sin(2*np.pi*5*t)             # inna częstotliwość

norma = np.linalg.norm(v1)           # wzór (2.9)
odleglosc = np.linalg.norm(v1 - v2)  # wzór (2.8)
iloczyn = np.vdot(v1, v2)            # iloczyn skalarny

kat = np.degrees(np.arccos(
    iloczyn / (norma * np.linalg.norm(v2))))
print(f"kąt między v1 i v2: {kat:.1f} stopni")
print(f"<v1|v3> = {np.vdot(v1, v3):.2e}")

# nierówność Schwarza (2.10)
print(abs(iloczyn) <= norma * np.linalg.norm(v2))

kąt między v1 i v2: 60.0 stopni
<v1|v3> = 6.66e-16
True


### Kod 2.10. Norma, kąt i ortogonalność funkcji obliczone symbolicznie

In [11]:
import sympy as sp

x = sp.symbols("x")

def iloczyn(f, g, a=-1, b=1):                 # wzór (2.15)
    return sp.integrate(f*g, (x, a, b))

def norma(f, a=-1, b=1):
    return sp.sqrt(iloczyn(f, f, a, b))

print(norma(sp.Integer(1)), norma(x), norma(x**2))
print(iloczyn(x, x**2))                       # funkcje ortogonalne
kat = sp.acos(iloczyn(x, x**3)/(norma(x)*norma(x**3)))
print(sp.simplify(sp.cos(kat)), round(float(kat*180/sp.pi), 2))

# funkcje trygonometryczne na przedziale [0, 2 pi]
m, n = sp.symbols("m n", integer=True, positive=True)
print(iloczyn(sp.sin(m*x), sp.cos(n*x), 0, 2*sp.pi),
      iloczyn(sp.sin(m*x), sp.sin(m*x), 0, 2*sp.pi),
      iloczyn(sp.sin(2*x), sp.sin(3*x), 0, 2*sp.pi))

sqrt(2) sqrt(6)/3 sqrt(10)/5
0
sqrt(21)/5 23.58
0 pi 0


### Kod 2.11. Wyszukiwanie wzorca w zaszumionym sygnale za pomocą unormowanego iloczynu skalarnego

In [12]:
import numpy as np

rng = np.random.default_rng(6)
n = np.arange(64)
wzorzec = np.exp(-((n - 32)/6.0)**2)*np.cos(0.9*(n - 32))

sygnal = 0.4*rng.normal(size=1000)
for polozenie, amplituda in ((200, 1.0), (640, 0.6)):
    sygnal[polozenie:polozenie + 64] += amplituda*wzorzec

podobienstwo = np.zeros(len(sygnal) - 64)
for k in range(len(podobienstwo)):
    fragment = sygnal[k:k + 64]
    podobienstwo[k] = (fragment @ wzorzec / np.linalg.norm(fragment)
                       / np.linalg.norm(wzorzec))

kolejnosc = np.argsort(podobienstwo)[::-1]
pierwsze = kolejnosc[0]
drugie = next(k for k in kolejnosc if abs(k - pierwsze) > 64)
print(pierwsze, round(podobienstwo[pierwsze], 2))
print(drugie, round(podobienstwo[drugie], 2))

639 0.44
200 0.44


### Kod 2.12. Ortonormalizacja Grama–Schmidta

In [13]:
import numpy as np

def gram_schmidt(V):
    """Wiersze macierzy V -> wiersze ortonormalne."""
    B = []
    for v in V.astype(float):
        u = v - sum(np.vdot(b, v) * b for b in B)
        B.append(u / np.linalg.norm(u))
    return np.array(B)

V = np.array([[3, 4], [4, 3]])
B = gram_schmidt(V)
print(B)                    # [[0.6 0.8] [0.8 -0.6]]
print(np.allclose(B @ B.T, np.eye(2)))

[[ 0.6  0.8]
 [ 0.8 -0.6]]
True


### Kod 2.13. Baza Haara w przestrzeni ośmiowymiarowej i rozwinięcie sygnału

In [14]:
import numpy as np

def baza_haara(n):
    """Wiersze: ortonormalna baza Haara w R^n."""
    H = [np.ones(n)]
    dl = n
    while dl > 1:
        for start in range(0, n, dl):
            h = np.zeros(n)
            h[start:start + dl//2] = 1
            h[start + dl//2:start + dl] = -1
            H.append(h)
        dl //= 2
    H = np.array(H)
    return H / np.linalg.norm(H, axis=1, keepdims=True)

H = baza_haara(8)
print(np.allclose(H @ H.T, np.eye(8)))   # ortonormalna

v = np.array([4., 6, 10, 12, 8, 6, 5, 5])
c = H @ v                                # wzór (2.19)
print(np.round(c, 3))
print(np.allclose(H.T @ c, v))           # wzór (2.18)

True
[19.799  2.828 -6.     2.    -1.414 -1.414  1.414  0.   ]
True


### Kod 2.14. Ortonormalizacja potęg: wielomiany Legendre'a

In [15]:
import sympy as sp

x = sp.symbols("x")
iloczyn = lambda f, g: sp.integrate(f*g, (x, -1, 1))

baza = []
for potega in (sp.Integer(1), x, x**2, x**3):
    u = potega - sum(iloczyn(e, potega)*e for e in baza)   # rzut
    baza.append(sp.simplify(u/sp.sqrt(iloczyn(u, u))))     # unormowanie

for n, e in enumerate(baza):
    wzorzec = sp.sqrt(sp.Rational(2*n + 1, 2))*sp.legendre(n, x)
    print(sp.expand(e), "|", sp.simplify(e - wzorzec) == 0)

G = sp.Matrix(4, 4, lambda i, j: iloczyn(baza[i], baza[j]))
print(G == sp.eye(4))                        # macierz Grama

sqrt(2)/2 | True
sqrt(6)*x/2 | True
3*sqrt(10)*x**2/4 - sqrt(10)/4 | True
5*sqrt(14)*x**3/4 - 3*sqrt(14)*x/4 | True
True


### Kod 2.15. Przybliżenie czterech funkcji szesnastoma wyrazami w trzech bazach

In [16]:
import numpy as np

N, K = 1024, 16
x = -1 + 2*(np.arange(N) + 0.5)/N              # siatka na (-1, 1)

def baza_legendre():
    V = np.polynomial.legendre.legvander(x, K - 1)
    return np.linalg.qr(V)[0].T                # ortonormalna na siatce

def baza_trygonometryczna():
    f = [np.ones(N)]
    for k in range(1, K//2 + 1):
        f += [np.cos(np.pi*k*x), np.sin(np.pi*k*x)]
    B = np.array(f[:K])
    return B/np.linalg.norm(B, axis=1, keepdims=True)

def baza_haara():
    f = [np.ones(N)]
    for j in range(4):                         # 1 + 1 + 2 + 4 + 8 = 16
        dl = N//2**j
        for k in range(2**j):
            h = np.zeros(N)
            h[k*dl:k*dl + dl//2], h[k*dl + dl//2:(k + 1)*dl] = 1, -1
            f.append(h)
    B = np.array(f)
    return B/np.linalg.norm(B, axis=1, keepdims=True)

funkcje = {"gładka": np.exp(x)*np.sin(3*x),
           "z załamaniem": np.abs(x - 0.2),
           "ze skokiem": np.where(x < 0.25, -1.0, 1.0),
           "okresowa": np.cos(3*np.pi*x) + 0.5*np.sin(5*np.pi*x)}
bazy = {"Legendre": baza_legendre(), "tryg.": baza_trygonometryczna(),
        "Haar": baza_haara()}

print(" "*14 + "".join(f"{b:>11s}" for b in bazy))
for nazwa, f in funkcje.items():
    bledy = []
    for B in bazy.values():
        przyblizenie = B.T @ (B @ f)           # rzut na 16 funkcji
        bledy.append(np.linalg.norm(f - przyblizenie)/np.linalg.norm(f))
    print(f"{nazwa:14s}" + "".join(f"{100*b:10.4f}%" for b in bledy))

                 Legendre      tryg.       Haar
gładka            0.0000%    4.0336%   10.2900%
z załamaniem      1.2005%    5.5245%    5.7760%
ze skokiem       19.7056%   22.4340%    0.0000%
okresowa          9.2773%    0.0000%   38.0608%


## Funkcje pomocnicze

Funkcje i dane, z których korzystają rozwiązania ćwiczeń.

In [17]:
def baza_haara(n):
    """Wiersze: ortonormalna baza Haara w R^n (n = potęga dwójki)."""
    H = [np.ones(n)]
    dl = n
    while dl > 1:
        for start in range(0, n, dl):
            h = np.zeros(n)
            h[start:start + dl//2] = 1
            h[start + dl//2:start + dl] = -1
            H.append(h)
        dl //= 2
    H = np.array(H)
    return H / np.linalg.norm(H, axis=1, keepdims=True)

## Ćwiczenia i rozwiązania

### Ćwiczenie 2.1
Sprawdź, że zbiór {1, 2, 3, 4} z mnożeniem modulo 5 jest grupą. Podaj element odwrotny do każdego elementu i wskaż generator.

### Rozwiązanie 2.1

In [18]:
import itertools
A = [1, 2, 3, 4]
mn = lambda a, b: (a * b) % 5
print("działanie wewnętrzne:", all(mn(a, b) in A for a in A for b in A))
print("łączność:", all(mn(mn(a, b), c) == mn(a, mn(b, c))
                       for a, b, c in itertools.product(A, repeat=3)))
print("odwrotności:", {a: next(b for b in A if mn(a, b) == 1) for a in A})
for g in A:
    potegi = [pow(g, k, 5) for k in range(1, 5)]
    print(g, potegi, "generator" if set(potegi) == set(A) else "")

działanie wewnętrzne: True
łączność: True
odwrotności: {1: 1, 2: 3, 3: 2, 4: 4}
1 [1, 1, 1, 1] 
2 [2, 4, 3, 1] generator
3 [3, 4, 2, 1] generator
4 [4, 1, 4, 1] 


### Ćwiczenie 2.2
Wyjaśnij, dlaczego zbiór {0, 1, 2, 3} z dodawaniem i mnożeniem modulo 4 nie jest ciałem.

### Rozwiązanie 2.2

In [19]:
for a in (1, 2, 3):
    odw = [b for b in (1, 2, 3) if (a*b) % 4 == 1]
    print(a, "odwrotność:", odw if odw else "brak")
print("2 * 2 mod 4 =", (2*2) % 4)   # dzielnik zera

1 odwrotność: [1]
2 odwrotność: brak
3 odwrotność: [3]
2 * 2 mod 4 = 0


### Ćwiczenie 2.3
Oblicz normę wektorów (1, 2, 2) i (2, 1, −2), ich iloczyn skalarny oraz kąt między nimi.

### Rozwiązanie 2.3

In [20]:
v1 = np.array([1., 2, 2]); v2 = np.array([2., 1, -2])
n1, n2, il = np.linalg.norm(v1), np.linalg.norm(v2), np.vdot(v1, v2)
print(n1, n2, il, np.degrees(np.arccos(il/(n1*n2))))

3.0 3.0 0.0 90.0


### Ćwiczenie 2.4
Metodą Grama–Schmidta zortonormalizuj wektory (1, 1, 0), (1, 0, 1), (0, 1, 1).

### Rozwiązanie 2.4

In [21]:
def gram_schmidt(V):
    B = []
    for v in V.astype(float):
        u = v - sum(np.vdot(b, v) * b for b in B)
        B.append(u / np.linalg.norm(u))
    return np.array(B)
B = gram_schmidt(np.array([[1, 1, 0], [1, 0, 1], [0, 1, 1]]))
print(np.round(B, 4))
print(np.allclose(B @ B.T, np.eye(3)))

[[ 0.7071  0.7071  0.    ]
 [ 0.4082 -0.4082  0.8165]
 [-0.5774  0.5774  0.5774]]
True


### Ćwiczenie 2.5
Rozwiń sygnał (2, 2, 2, 2, 6, 6, 0, 4) w bazie Haara. Które współczynniki są zerami i dlaczego? Sprawdź równość Parsevala.

### Rozwiązanie 2.5

In [22]:
H = baza_haara(8)                      # funkcja zdefiniowana w części „Funkcje pomocnicze"
v = np.array([2., 2, 2, 2, 6, 6, 0, 4])
c = H @ v
print(np.round(c, 3))
print(np.sum(v**2), round(float(np.sum(c**2)), 6))
# Współczynnik znika, gdy sygnał jest stały na nośniku wektora bazy.

[ 8.485 -2.828  0.     4.     0.     0.     0.    -2.828]
104.0 104.0


### Ćwiczenie 2.6
Zbuduj macierz filtru okresowego o współczynnikach (1, −1) dla N = 8 i wyznacz jej wartości własne. Porównaj je z dyskretną transformatą Fouriera współczynników. Która zespolona sinusoida jest przez ten filtr zerowana?

### Rozwiązanie 2.6

In [23]:
N = 8
S = np.roll(np.eye(N), 1, axis=0)
C = np.eye(N) - S
wartosci = np.fft.fft([1, -1], N)
n = np.arange(N)
for k in range(N):
    chi = np.exp(2j*np.pi*k*n/N)
    print(k, np.allclose(C @ chi, wartosci[k]*chi), np.round(wartosci[k], 3))
# Wartość własna dla k = 0 jest zerem: filtr różnicowy usuwa składową stałą.

0 True 0j
1 True (0.293+0.707j)
2 True (1+1j)
3 True (1.707+0.707j)
4 True (2+0j)
5 True (1.707-0.707j)
6 True (1-1j)
7 True (0.293-0.707j)


### Ćwiczenie 2.7
Metodą Grama–Schmidta w SymPy zortonormalizuj funkcje 1, x, x² na przedziale [0, 1]. Sprawdź macierz Grama wyniku.

### Rozwiązanie 2.7

In [24]:
import sympy as sp
x = sp.symbols("x")
il = lambda f, g: sp.integrate(f*g, (x, 0, 1))
baza = []
for potega in (sp.Integer(1), x, x**2):
    u = potega - sum(il(e, potega)*e for e in baza)
    baza.append(sp.simplify(u/sp.sqrt(il(u, u))))
for e in baza:
    print(sp.expand(e))
print(sp.Matrix(3, 3, lambda i, j: il(baza[i], baza[j])) == sp.eye(3))

1
2*sqrt(3)*x - sqrt(3)
6*sqrt(5)*x**2 - 6*sqrt(5)*x + sqrt(5)
True


### Ćwiczenie 2.8
Powtórz obliczenia z kodu 2.15 dla funkcji ze skokiem w punkcie x = 0,3, który nie leży na granicy przedziałów diadycznych. Jak zmienia się błąd w bazie Haara dla K = 16, 64 i 256 funkcji?

### Rozwiązanie 2.8

In [25]:
N = 1024
x = -1 + 2*(np.arange(N) + 0.5)/N
f = np.where(x < 0.3, -1.0, 1.0)
def haar(K):
    fun = [np.ones(N)]
    for j in range(int(np.log2(K))):
        dl = N//2**j
        for k in range(2**j):
            h = np.zeros(N)
            h[k*dl:k*dl + dl//2], h[k*dl + dl//2:(k + 1)*dl] = 1, -1
            fun.append(h)
    B = np.array(fun)
    return B/np.linalg.norm(B, axis=1, keepdims=True)
for K in (16, 64, 256):
    B = haar(K)
    blad = np.linalg.norm(f - B.T @ (B @ f))/np.linalg.norm(f)
    niezerowe = np.sum(np.abs(B @ f) > 1e-9)
    print(f"K = {K:3d}: błąd {100*blad:5.2f}%, niezerowych współczynników: {niezerowe}")
# Błąd maleje powoli (jak pierwiastek z długości najkrótszego odcinka),
# ale niezerowych współczynników jest tylko po jednym na poziom:
# skok „widzą” wyłącznie funkcje, których nośnik go zawiera.

K =  16: błąd 24.56%, niezerowych współczynników: 5
K =  64: błąd 12.10%, niezerowych współczynników: 7
K = 256: błąd  6.25%, niezerowych współczynników: 9
